# Chapter 2.1: Anatomy of an LLM Serving System -- Request Lifecycle

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the complete lifecycle** of a request through an LLM serving system, from API reception to response delivery
2. **Explain each processing stage** -- validation, tokenization, scheduling, prefill, decode, detokenization, and streaming
3. **Distinguish prefill from decode** and understand why they have fundamentally different compute characteristics
4. **Implement a toy LLM server** that demonstrates all lifecycle stages
5. **Reason about connection management** concerns: keep-alive, timeouts, and cancellation

## Prerequisites

- Basic understanding of Transformer architecture (Part 1)
- Familiarity with HTTP APIs and async Python
- Understanding of tokenization (BPE, SentencePiece)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part2/chapter_2.1_request_lifecycle.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part2/chapter_2.1_request_lifecycle.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. The Big Picture: End-to-End Request Flow

When you send a prompt like *"Explain quantum computing in simple terms"* to an LLM API, what actually happens under the hood? The answer involves a surprisingly complex pipeline that modern serving systems like vLLM and SGLang have carefully optimized.

Let's first visualize the complete flow, then dive deep into each stage.

In [ ]:
## Figure: Full Request Lifecycle Flowchart
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(18, 6))
ax.set_xlim(0, 22)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('Complete Request Lifecycle in an LLM Serving System', fontsize=16, fontweight='bold')

BLUE, GREEN, ORANGE, RED, PURPLE = '#4A90D9', '#27AE60', '#F39C12', '#E74C3C', '#8E44AD'
GRAY = '#95A5A6'

stages = [
    (1, 3, 'Client\nRequest', PURPLE),
    (3.5, 3, 'API\nServer', GRAY),
    (6, 3, 'Tokenizer', ORANGE),
    (8.5, 3, 'Scheduler', RED),
    (11, 3.8, 'Prefill', RED),
    (13, 3.8, 'KV-Cache', GREEN),
    (15, 3.8, 'Decode\nLoop', BLUE),
    (17.5, 3, 'Detokenizer', ORANGE),
    (20, 3, 'Stream\nResponse', PURPLE),
]

for x, y, label, color in stages:
    ax.add_patch(patches.FancyBboxPatch((x-0.9, y-0.6), 1.8, 1.2, boxstyle="round,pad=0.12",
                 facecolor=color, alpha=0.8, edgecolor='black', linewidth=1.5))
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# Arrows between stages
arrow_pairs = [(1.9,3,2.6,3), (4.4,3,5.1,3), (6.9,3,7.6,3), (9.4,3,10.1,3.5),
               (11.9,3.8,12.1,3.8), (13.9,3.8,14.1,3.8), (15.9,3.5,16.6,3), (18.4,3,19.1,3)]
for x1,y1,x2,y2 in arrow_pairs:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1), arrowprops=dict(arrowstyle='->', lw=2, color='#2C3E50'))

# Decode loop arrow back
ax.annotate('', xy=(14.1, 4.5), xytext=(15.9, 4.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color=BLUE, connectionstyle='arc3,rad=-0.3'))
ax.text(15, 5.2, 'Repeat until\nEOS or max_len', fontsize=8, ha='center', color=BLUE, style='italic')

# Phase labels
ax.add_patch(patches.FancyBboxPatch((0.2, 1.2), 8.5, 0.6, boxstyle="round,pad=0.1",
             facecolor='lightyellow', alpha=0.5, edgecolor=ORANGE))
ax.text(4.5, 1.5, 'Request Processing (CPU)', ha='center', fontsize=9, fontweight='bold', color=ORANGE)

ax.add_patch(patches.FancyBboxPatch((9.5, 1.2), 7.5, 0.6, boxstyle="round,pad=0.1",
             facecolor='#EBF5FB', alpha=0.5, edgecolor=BLUE))
ax.text(13.25, 1.5, 'Model Execution (GPU)', ha='center', fontsize=9, fontweight='bold', color=BLUE)

ax.add_patch(patches.FancyBboxPatch((16.5, 1.2), 5, 0.6, boxstyle="round,pad=0.1",
             facecolor='#F5EEF8', alpha=0.5, edgecolor=PURPLE))
ax.text(19, 1.5, 'Response (CPU + Network)', ha='center', fontsize=9, fontweight='bold', color=PURPLE)

plt.tight_layout()
plt.show()
print("The request flows: Client -> API -> Tokenize -> Schedule -> Prefill -> Decode Loop -> Detokenize -> Stream")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 20)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('End-to-End Request Lifecycle in an LLM Serving System', fontsize=16, fontweight='bold', pad=20)

# Define stages with positions and colors
stages = [
    (1, 10, 'API\nReception', '#3498db', 'HTTP POST\n/v1/chat/completions'),
    (5, 10, 'Request\nValidation', '#2ecc71', 'Auth, params,\ncontext length'),
    (9, 10, 'Tokenization', '#e74c3c', 'Text -> Token IDs\n(BPE/SPM)'),
    (13, 10, 'Scheduling', '#9b59b6', 'Queue management\nPriority/FCFS'),
    (17, 10, 'GPU Memory\nAllocation', '#f39c12', 'KV-cache blocks\nreservation'),
    (3, 6, 'Prefill\n(1st Forward Pass)', '#e67e22', 'Process ALL input\ntokens at once'),
    (8, 6, 'Decode Loop\n(Autoregressive)', '#1abc9c', 'Generate 1 token\nper iteration'),
    (13, 6, 'Detokenization', '#e74c3c', 'Token IDs -> Text\n(streaming)'),
    (17, 6, 'Response\nDelivery', '#3498db', 'SSE stream or\nfull JSON response'),
]

for x, y, label, color, detail in stages:
    box = FancyBboxPatch((x-1.3, y-0.8), 2.6, 1.6, 
                          boxstyle="round,pad=0.1", 
                          facecolor=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(x, y-1.3, detail, ha='center', va='top', fontsize=7, color='#2c3e50', style='italic')

# Draw arrows
arrow_style = dict(arrowstyle='->', color='#2c3e50', lw=2)
# Top row arrows
for x1, x2 in [(2.3, 3.7), (6.3, 7.7), (10.3, 11.7), (14.3, 15.7)]:
    ax.annotate('', xy=(x2, 10), xytext=(x1, 10), arrowprops=arrow_style)
# Down arrow from GPU Memory Alloc to Prefill row
ax.annotate('', xy=(17, 7.2), xytext=(17, 9.2), arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2, connectionstyle='arc3,rad=0.3'))
ax.annotate('', xy=(4.3, 6), xytext=(15.7, 7), arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2, connectionstyle='arc3,rad=0.5'))

# Bottom row arrows
for x1, x2 in [(4.3, 6.7), (9.3, 11.7), (14.3, 15.7)]:
    ax.annotate('', xy=(x2, 6), xytext=(x1, 6), arrowprops=arrow_style)

# Decode loop arrow
ax.annotate('', xy=(7, 5.0), xytext=(9, 5.0), 
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2, connectionstyle='arc3,rad=-0.5'))
ax.text(8, 4.2, 'repeat until\nEOS or max_tokens', ha='center', va='top', fontsize=7, color='#e74c3c', fontweight='bold')

# Time annotations
ax.annotate('', xy=(1, 3), xytext=(17, 3), arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.text(9, 2.5, 'Total Latency (Time To Last Token - TTLT)', ha='center', fontsize=10, color='gray')
ax.annotate('', xy=(1, 2), xytext=(8, 2), arrowprops=dict(arrowstyle='<->', color='#e67e22', lw=1.5))
ax.text(4.5, 1.5, 'Time To First Token (TTFT)', ha='center', fontsize=9, color='#e67e22')

plt.tight_layout()
plt.show()

## 2. Stage 1: API Reception

The journey begins when an HTTP request arrives at the serving endpoint. Modern LLM servers typically use an **async web framework** (FastAPI, aiohttp) to handle potentially thousands of concurrent connections.

### Key Design Decisions at This Stage

| Concern | Design Choice | Why |
|---------|--------------|-----|
| Concurrency model | Async I/O (asyncio) | LLM inference is GPU-bound; CPU threads would be wasted waiting |
| Protocol | HTTP/1.1 with SSE or HTTP/2 | SSE for streaming; HTTP/2 for multiplexing |
| Serialization | JSON (OpenAI-compatible) | Ecosystem compatibility |
| Connection limits | Max concurrent connections | Prevent server overload |

### How vLLM and SGLang Handle This

- **vLLM**: Uses FastAPI with uvicorn. The `AsyncLLMEngine` receives requests and puts them into an internal queue.
- **SGLang**: Uses a custom async runtime with a `TokenizerManager` that handles request reception and tokenization.

In [ ]:
# Simulating the API reception layer
import asyncio
import json
import time
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
from enum import Enum
import uuid

class RequestStatus(Enum):
    PENDING = "pending"
    VALIDATING = "validating"
    TOKENIZING = "tokenizing"
    QUEUED = "queued"
    PREFILLING = "prefilling"
    DECODING = "decoding"
    DETOKENIZING = "detokenizing"
    COMPLETE = "complete"
    ERROR = "error"
    CANCELLED = "cancelled"

@dataclass
class LLMRequest:
    """Represents a single LLM inference request as it flows through the system."""
    request_id: str
    prompt: str
    max_tokens: int = 100
    temperature: float = 1.0
    stop_sequences: List[str] = field(default_factory=list)
    stream: bool = False
    
    # Internal state - populated as request flows through pipeline
    status: RequestStatus = RequestStatus.PENDING
    token_ids: Optional[List[int]] = None
    generated_token_ids: List[int] = field(default_factory=list)
    generated_text: str = ""
    
    # Timing information
    arrival_time: float = 0.0
    validation_time: float = 0.0
    tokenization_time: float = 0.0
    scheduling_time: float = 0.0
    prefill_start_time: float = 0.0
    first_token_time: float = 0.0
    completion_time: float = 0.0
    
    def __post_init__(self):
        self.arrival_time = time.time()
    
    @property
    def ttft(self) -> Optional[float]:
        """Time to First Token"""
        if self.first_token_time > 0:
            return self.first_token_time - self.arrival_time
        return None
    
    @property
    def total_latency(self) -> Optional[float]:
        """Total end-to-end latency"""
        if self.completion_time > 0:
            return self.completion_time - self.arrival_time
        return None

# Create a sample request (simulating what arrives from HTTP)
sample_request_body = {
    "model": "llama-3-8b",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    "max_tokens": 150,
    "temperature": 0.7,
    "stream": True
}

print("=== Incoming HTTP Request Body ===")
print(json.dumps(sample_request_body, indent=2))
print(f"\nRequest size: {len(json.dumps(sample_request_body))} bytes")

## 3. Stage 2: Request Validation

Before any expensive computation, the server validates the incoming request. This is a critical gatekeeping step that prevents wasted GPU cycles.

### Validation Checks

1. **Authentication/Authorization**: Is the API key valid? Does this user have access?
2. **Schema Validation**: Are all required fields present? Are types correct?
3. **Parameter Bounds**: Is `temperature` in [0, 2]? Is `max_tokens` within model limits?
4. **Context Length Check**: Will the prompt + max_tokens exceed the model's context window?
5. **Model Availability**: Is the requested model loaded and ready?
6. **Rate Limiting**: Has this user exceeded their request rate?

In [ ]:
class RequestValidator:
    """Validates incoming LLM requests before they enter the processing pipeline."""
    
    def __init__(self, max_context_length: int = 8192, 
                 available_models: List[str] = None):
        self.max_context_length = max_context_length
        self.available_models = available_models or ["llama-3-8b", "llama-3-70b"]
        self.rate_limits: Dict[str, List[float]] = {}  # api_key -> [timestamps]
        self.max_requests_per_minute = 60
    
    def validate(self, request_body: dict, api_key: str = "test-key") -> tuple:
        """Validate request. Returns (is_valid, error_message_or_none)."""
        errors = []
        
        # 1. Model availability
        model = request_body.get("model")
        if model not in self.available_models:
            errors.append(f"Model '{model}' not found. Available: {self.available_models}")
        
        # 2. Required fields
        if "messages" not in request_body and "prompt" not in request_body:
            errors.append("Either 'messages' or 'prompt' is required")
        
        # 3. Parameter bounds
        temp = request_body.get("temperature", 1.0)
        if not (0 <= temp <= 2.0):
            errors.append(f"temperature must be in [0, 2], got {temp}")
        
        max_tokens = request_body.get("max_tokens", 100)
        if max_tokens < 1 or max_tokens > self.max_context_length:
            errors.append(f"max_tokens must be in [1, {self.max_context_length}], got {max_tokens}")
        
        # 4. Context length (rough estimate before tokenization)
        if "messages" in request_body:
            total_chars = sum(len(m.get("content", "")) for m in request_body["messages"])
            estimated_tokens = total_chars // 4  # rough: ~4 chars per token
            if estimated_tokens + max_tokens > self.max_context_length:
                errors.append(
                    f"Estimated {estimated_tokens} prompt tokens + {max_tokens} max_tokens "
                    f"exceeds context length {self.max_context_length}"
                )
        
        # 5. Rate limiting
        now = time.time()
        if api_key not in self.rate_limits:
            self.rate_limits[api_key] = []
        # Remove timestamps older than 60 seconds
        self.rate_limits[api_key] = [t for t in self.rate_limits[api_key] if now - t < 60]
        if len(self.rate_limits[api_key]) >= self.max_requests_per_minute:
            errors.append(f"Rate limit exceeded: {self.max_requests_per_minute} requests/minute")
        self.rate_limits[api_key].append(now)
        
        return (len(errors) == 0, errors)

# Test validation
validator = RequestValidator()

# Valid request
is_valid, errors = validator.validate(sample_request_body)
print(f"Valid request: is_valid={is_valid}, errors={errors}")

# Invalid requests
bad_requests = [
    {"model": "gpt-4", "messages": [{"role": "user", "content": "hi"}]},  # wrong model
    {"model": "llama-3-8b"},  # missing messages
    {"model": "llama-3-8b", "messages": [{"role": "user", "content": "hi"}], "temperature": 5.0},  # bad temp
]

for i, req in enumerate(bad_requests):
    is_valid, errors = validator.validate(req)
    print(f"\nBad request {i+1}: is_valid={is_valid}")
    for e in errors:
        print(f"  - {e}")

## 4. Stage 3: Tokenization

Once validated, the text must be converted to **token IDs** that the model can process. This is a CPU-bound operation that involves:

1. **Chat template application**: Converting the list of messages into a single formatted string
2. **Tokenization**: Using BPE (Byte-Pair Encoding) or SentencePiece to convert text to token IDs
3. **Special token insertion**: Adding BOS (Beginning of Sequence), EOS, and role markers

### Why Tokenization Matters for Serving

- The number of tokens determines **KV-cache memory** needed
- The number of tokens determines **prefill compute cost** (quadratic in attention)
- Accurate token counting is needed for **context length validation**
- Tokenization must be **exactly the same** as what the model was trained with

In [ ]:
class SimpleTokenizer:
    """A simplified tokenizer that demonstrates the concept.
    Real tokenizers (BPE, SentencePiece) are much more sophisticated."""
    
    def __init__(self):
        # Simulated vocabulary (real Llama-3 has ~128K tokens)
        self.vocab = {}
        self.reverse_vocab = {}
        
        # Build a simple word-level vocabulary
        words = [
            "<bos>", "<eos>", "<pad>", "<|system|>", "<|user|>", "<|assistant|>",
            "the", "a", "an", "is", "are", "was", "in", "of", "to", "and",
            "you", "i", "it", "that", "this", "for", "with", "on", "at",
            "quantum", "computing", "computer", "simple", "terms", "explain",
            "helpful", "assistant", "hello", "world", "what", "how", "why",
            "bit", "qubit", "super", "position", "entangle", "ment",
            "class", "ical", "uses", "bits", "which", "can", "be", "either",
            "0", "or", "1", ".", ",", "?", "!", " ", "\n",
        ]
        for i, w in enumerate(words):
            self.vocab[w] = i
            self.reverse_vocab[i] = w
        self.unk_id = len(self.vocab)
        self.bos_id = 0
        self.eos_id = 1
    
    def apply_chat_template(self, messages: List[dict]) -> str:
        """Convert chat messages to a single formatted string."""
        formatted = ""
        for msg in messages:
            role = msg["role"]
            content = msg["content"]
            formatted += f"<|{role}|>\n{content}\n"
        formatted += "<|assistant|>\n"  # Prompt the model to generate
        return formatted
    
    def encode(self, text: str) -> List[int]:
        """Tokenize text into token IDs (simplified word-level)."""
        tokens = [self.bos_id]  # Always start with BOS
        words = text.lower().split()
        for word in words:
            if word in self.vocab:
                tokens.append(self.vocab[word])
            else:
                tokens.append(self.unk_id)  # Unknown token
        return tokens
    
    def decode(self, token_ids: List[int]) -> str:
        """Convert token IDs back to text."""
        words = []
        for tid in token_ids:
            if tid in self.reverse_vocab:
                words.append(self.reverse_vocab[tid])
            else:
                words.append(f"[UNK:{tid}]")
        return " ".join(words)

# Demonstrate tokenization
tokenizer = SimpleTokenizer()

# Apply chat template
messages = sample_request_body["messages"]
formatted_prompt = tokenizer.apply_chat_template(messages)
print("=== Chat Template Output ===")
print(formatted_prompt)

# Tokenize
token_ids = tokenizer.encode(formatted_prompt)
print(f"\n=== Token IDs ({len(token_ids)} tokens) ===")
print(token_ids)

# Show token-by-token mapping
print("\n=== Token Mapping ===")
for tid in token_ids[:15]:
    text = tokenizer.reverse_vocab.get(tid, f"[UNK:{tid}]")
    print(f"  ID {tid:3d} -> '{text}'")
print(f"  ... ({len(token_ids) - 15} more tokens)")

## 5. Stage 4: Scheduling -- When Does a Request Get GPU Time?

After tokenization, the request enters the **scheduler**. This is the brain of the serving system. The scheduler must decide:

1. **Which requests to run** in the current iteration (the "running batch")
2. **Which requests to wait** (the "waiting queue")
3. **Whether to preempt** running requests to admit new ones

### The Continuous Batching Revolution

Traditional serving systems used **static batching**: collect N requests, run them together, wait for ALL to finish, then start the next batch. This is wasteful because short requests must wait for long ones.

**Continuous batching** (also called **iteration-level scheduling**) allows:
- New requests to join the batch at any iteration
- Finished requests to leave immediately
- Much higher GPU utilization

In [ ]:
# Visualize Static vs Continuous Batching

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# === Static Batching ===
ax = axes[0]
ax.set_title('Static Batching (Naive)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time (iterations)')
ax.set_ylabel('Request')

# Batch 1: 3 requests with different lengths
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
requests_static = [
    (0, 8, 'Req A (8 tokens)'),   # arrives at t=0, needs 8 decode steps
    (0, 3, 'Req B (3 tokens)'),   # arrives at t=0, needs 3 decode steps
    (0, 6, 'Req C (6 tokens)'),   # arrives at t=0, needs 6 decode steps
    (8, 4, 'Req D (4 tokens)'),   # arrives at t=2, but must wait for batch 1
    (8, 5, 'Req E (5 tokens)'),   # arrives at t=3, but must wait for batch 1
]

for i, (start, length, label) in enumerate(requests_static):
    # Active computation
    ax.barh(i, length, left=start, height=0.6, color=colors[i], alpha=0.8, edgecolor='black')
    # Wasted time (padding for shorter requests in batch)
    if i < 3:  # Batch 1
        waste = 8 - length
        if waste > 0:
            ax.barh(i, waste, left=start+length, height=0.6, color='lightgray', alpha=0.5, 
                    edgecolor='black', linestyle='--', hatch='//')
    ax.text(start + 0.2, i, label, va='center', fontsize=8, fontweight='bold')

ax.set_yticks(range(5))
ax.set_yticklabels([f'Slot {i}' for i in range(5)])
ax.axvline(x=8, color='red', linestyle='--', alpha=0.5, label='Batch boundary')
ax.legend(loc='upper right')
# Add annotation for wasted GPU cycles
ax.annotate('Wasted GPU cycles!', xy=(5, 1), fontsize=10, color='red', fontweight='bold')

# === Continuous Batching ===
ax = axes[1]
ax.set_title('Continuous Batching (vLLM/SGLang)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time (iterations)')
ax.set_ylabel('Request')

requests_continuous = [
    (0, 8, 'Req A'),
    (0, 3, 'Req B'),
    (0, 6, 'Req C'),
    (3, 4, 'Req D (joins when B finishes!)'),  # joins right after B leaves
    (6, 5, 'Req E (joins when C finishes!)'),   # joins right after C leaves
]

for i, (start, length, label) in enumerate(requests_continuous):
    row = i if i < 3 else (1 if i == 3 else 2)  # D reuses B's slot, E reuses C's
    ax.barh(row, length, left=start, height=0.6, color=colors[i], alpha=0.8, edgecolor='black')
    ax.text(start + 0.2, row, label, va='center', fontsize=8, fontweight='bold')

ax.set_yticks(range(3))
ax.set_yticklabels([f'Slot {i}' for i in range(3)])
ax.annotate('No wasted cycles!\nNew requests join immediately', 
            xy=(4, 2.5), fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Stage 5: Prefill -- The First Forward Pass

The **prefill** stage is where the input tokens are processed for the first time. This is fundamentally different from decoding:

### Prefill vs Decode: A Critical Distinction

| Property | Prefill | Decode |
|----------|---------|--------|
| Input size | All prompt tokens at once | 1 token per step |
| Compute pattern | **Compute-bound** (matrix-matrix multiply) | **Memory-bound** (matrix-vector multiply) |
| Attention | Full self-attention over all tokens | Attention over all KV-cache + 1 new token |
| KV-cache | Generated from scratch | Appended to |
| GPU utilization | High (large matrices) | Low (small operations, memory-bottlenecked) |
| Latency impact | Dominates TTFT | Dominates per-token latency |

### What Happens During Prefill

For each layer in the Transformer:
1. Compute Q, K, V projections for ALL input tokens
2. Compute self-attention (this is O(n^2) in sequence length!)
3. Store K and V in the KV-cache for future decode steps
4. Apply feed-forward network

The result: KV-cache is fully populated, and we have logits for the next token.

In [ ]:
import numpy as np

class SimulatedTransformerLayer:
    """Simulates the computation pattern of a single transformer layer."""
    
    def __init__(self, hidden_dim=4096, num_heads=32, head_dim=128):
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.kv_cache_k = None  # Will store key cache
        self.kv_cache_v = None  # Will store value cache
    
    def prefill(self, hidden_states: np.ndarray) -> dict:
        """Prefill: process all tokens at once.
        hidden_states shape: [seq_len, hidden_dim]
        """
        seq_len = hidden_states.shape[0]
        
        # Simulate QKV projection: [seq_len, hidden_dim] @ [hidden_dim, 3*hidden_dim]
        qkv_flops = seq_len * self.hidden_dim * 3 * self.hidden_dim
        
        # Simulate attention: Q @ K^T -> [seq_len, seq_len], then @ V -> [seq_len, head_dim]
        attn_flops = 2 * seq_len * seq_len * self.hidden_dim  # QK^T and attn@V
        
        # Simulate output projection: [seq_len, hidden_dim] @ [hidden_dim, hidden_dim]
        out_flops = seq_len * self.hidden_dim * self.hidden_dim
        
        # Simulate FFN: two linear layers, typically 4x expansion
        ffn_flops = 2 * seq_len * self.hidden_dim * 4 * self.hidden_dim
        
        total_flops = qkv_flops + attn_flops + out_flops + ffn_flops
        
        # KV-cache memory: 2 * seq_len * hidden_dim * sizeof(float16)
        kv_cache_bytes = 2 * seq_len * self.hidden_dim * 2  # fp16
        
        return {
            'operation': 'prefill',
            'seq_len': seq_len,
            'total_flops': total_flops,
            'qkv_flops': qkv_flops,
            'attention_flops': attn_flops,
            'ffn_flops': ffn_flops,
            'kv_cache_bytes': kv_cache_bytes,
            'compute_pattern': 'matrix-matrix (GEMM) - compute bound',
        }
    
    def decode_step(self, current_kv_len: int) -> dict:
        """Decode: process a single new token."""
        seq_len = 1  # Only processing ONE token
        
        # QKV projection: [1, hidden_dim] @ [hidden_dim, 3*hidden_dim]
        qkv_flops = seq_len * self.hidden_dim * 3 * self.hidden_dim
        
        # Attention: Q(1) @ K^T(current_kv_len) -> [1, kv_len], then @ V
        attn_flops = 2 * seq_len * current_kv_len * self.hidden_dim
        
        # Output projection
        out_flops = seq_len * self.hidden_dim * self.hidden_dim
        
        # FFN
        ffn_flops = 2 * seq_len * self.hidden_dim * 4 * self.hidden_dim
        
        total_flops = qkv_flops + attn_flops + out_flops + ffn_flops
        
        # Only need to read model weights + KV-cache (memory bound!)
        model_weight_bytes = (3 * self.hidden_dim * self.hidden_dim + 
                             self.hidden_dim * self.hidden_dim +
                             2 * self.hidden_dim * 4 * self.hidden_dim) * 2  # fp16
        kv_read_bytes = 2 * current_kv_len * self.hidden_dim * 2  # read existing KV
        
        return {
            'operation': 'decode',
            'kv_cache_len': current_kv_len,
            'total_flops': total_flops,
            'memory_read_bytes': model_weight_bytes + kv_read_bytes,
            'compute_pattern': 'matrix-vector (GEMV) - memory bound',
            'arithmetic_intensity': total_flops / (model_weight_bytes + kv_read_bytes),
        }

# Compare prefill vs decode
layer = SimulatedTransformerLayer()

# Prefill with 512 tokens
hidden = np.random.randn(512, 4096).astype(np.float16)
prefill_stats = layer.prefill(hidden)

# Decode with 512 existing KV-cache entries
decode_stats = layer.decode_step(512)

print("=== Prefill (512 tokens, 1 layer) ===")
print(f"  Total FLOPs: {prefill_stats['total_flops']/1e9:.2f} GFLOPs")
print(f"  KV-cache generated: {prefill_stats['kv_cache_bytes']/1024:.1f} KB")
print(f"  Pattern: {prefill_stats['compute_pattern']}")

print(f"\n=== Decode (1 token, KV-cache len=512, 1 layer) ===")
print(f"  Total FLOPs: {decode_stats['total_flops']/1e9:.4f} GFLOPs")
print(f"  Memory read: {decode_stats['memory_read_bytes']/1024/1024:.2f} MB")
print(f"  Arithmetic intensity: {decode_stats['arithmetic_intensity']:.2f} FLOPs/byte")
print(f"  Pattern: {decode_stats['compute_pattern']}")

print(f"\n=== Ratio ===")
print(f"  Prefill FLOPs / Decode FLOPs = {prefill_stats['total_flops']/decode_stats['total_flops']:.0f}x")
print(f"  (Prefill does {512}x more useful work per GPU call!)")

In [ ]:
# Visualize compute intensity: Prefill vs Decode
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: FLOPs comparison across different prompt lengths
ax = axes[0]
prompt_lengths = [64, 128, 256, 512, 1024, 2048, 4096]
prefill_flops = []
decode_flops = []

for n in prompt_lengths:
    h = np.random.randn(n, 4096).astype(np.float16)
    pf = layer.prefill(h)
    dc = layer.decode_step(n)
    prefill_flops.append(pf['total_flops'] / 1e9)
    decode_flops.append(dc['total_flops'] / 1e9)

x = np.arange(len(prompt_lengths))
width = 0.35
ax.bar(x - width/2, prefill_flops, width, label='Prefill (all tokens)', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, decode_flops, width, label='Decode (1 token)', color='#3498db', alpha=0.8)
ax.set_xlabel('Prompt Length')
ax.set_ylabel('GFLOPs (per layer)')
ax.set_title('Prefill vs Decode: Compute Cost')
ax.set_xticks(x)
ax.set_xticklabels(prompt_lengths)
ax.legend()
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Where time is spent (pie chart)
ax = axes[1]
# For a typical request: 512 token prompt, 200 token generation
prefill_cost = layer.prefill(np.random.randn(512, 4096).astype(np.float16))['total_flops']
decode_total = sum(layer.decode_step(512 + i)['total_flops'] for i in range(200))

sizes = [prefill_cost, decode_total]
labels = [f'Prefill\n(512 tokens)\n{prefill_cost/1e9:.1f} GFLOPs', 
          f'Decode\n(200 steps)\n{decode_total/1e9:.1f} GFLOPs']
ax.pie(sizes, labels=labels, colors=['#e74c3c', '#3498db'], 
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax.set_title('Compute Distribution\n(512-token prompt, 200-token generation)')

plt.tight_layout()
plt.show()

## 7. Stage 6: Decode Iterations -- One Token at a Time

After prefill populates the KV-cache, the model enters the **decode loop**. Each iteration:

1. Takes the previously generated token as input
2. Computes attention against the entire KV-cache
3. Produces logits for the next token
4. Applies sampling (temperature, top-k, top-p) to select the next token
5. Appends the new K, V to the cache
6. Checks stopping conditions (EOS token, max_tokens, stop sequence)

### Sampling Strategies

The raw output of the model is a probability distribution over the vocabulary. How we select from this distribution affects generation quality:

- **Greedy (temperature=0)**: Always pick the most likely token. Deterministic but can be repetitive.
- **Temperature sampling**: Scale logits by 1/T before softmax. Higher T = more random.
- **Top-k**: Only consider the k most likely tokens.
- **Top-p (nucleus)**: Only consider tokens whose cumulative probability is <= p.

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def sample_token(logits: np.ndarray, temperature: float = 1.0, 
                 top_k: int = 0, top_p: float = 1.0) -> int:
    """Sample a token from logits using various strategies."""
    # Apply temperature
    if temperature == 0:
        return int(np.argmax(logits))
    
    scaled_logits = logits / temperature
    
    # Apply top-k
    if top_k > 0:
        top_k_indices = np.argsort(scaled_logits)[-top_k:]
        mask = np.full_like(scaled_logits, -np.inf)
        mask[top_k_indices] = scaled_logits[top_k_indices]
        scaled_logits = mask
    
    probs = softmax(scaled_logits)
    
    # Apply top-p (nucleus sampling)
    if top_p < 1.0:
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs = probs[sorted_indices]
        cumsum = np.cumsum(sorted_probs)
        mask = cumsum <= top_p
        mask[0] = True  # Always keep at least one token
        # Zero out tokens below the top-p threshold
        filtered = np.zeros_like(probs)
        filtered[sorted_indices[mask]] = probs[sorted_indices[mask]]
        probs = filtered / filtered.sum()
    
    return int(np.random.choice(len(probs), p=probs))

# Demonstrate sampling strategies
np.random.seed(42)
vocab_size = 50
raw_logits = np.random.randn(vocab_size) * 2  # Simulated model output

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

strategies = [
    ('Greedy (temp=0)', {'temperature': 0.001}),  # Near-zero to avoid div by 0
    ('Temperature=0.7', {'temperature': 0.7}),
    ('Top-k=10, temp=1.0', {'temperature': 1.0, 'top_k': 10}),
    ('Top-p=0.9, temp=1.0', {'temperature': 1.0, 'top_p': 0.9}),
]

for ax, (name, params) in zip(axes.flat, strategies):
    # Sample 1000 times to show distribution
    samples = [sample_token(raw_logits.copy(), **params) for _ in range(1000)]
    counts = np.bincount(samples, minlength=vocab_size)
    
    ax.bar(range(vocab_size), counts, color='#3498db', alpha=0.7)
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Token ID')
    ax.set_ylabel('Sample Count (out of 1000)')
    ax.set_xlim(-1, vocab_size)

plt.suptitle('Effect of Sampling Strategy on Token Selection Distribution', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Stage 7: Detokenization and Streaming

As tokens are generated, they must be converted back to text and streamed to the client.

### The Streaming Challenge

Detokenization is not always straightforward because:
- Some tokens represent partial words (subword tokenization)
- Unicode characters may span multiple tokens
- We need **incremental detokenization**: only send the new text delta, not re-decode everything

### Server-Sent Events (SSE)

LLM APIs use SSE to stream tokens to the client as they are generated. The format follows the OpenAI API convention:

```
data: {"choices": [{"delta": {"content": "Quantum"}}]}
data: {"choices": [{"delta": {"content": " computing"}}]}
data: {"choices": [{"delta": {"content": " is"}}]}
...
data: [DONE]
```

In [ ]:
class IncrementalDetokenizer:
    """Handles incremental detokenization for streaming."""
    
    def __init__(self, tokenizer: SimpleTokenizer):
        self.tokenizer = tokenizer
        self.all_token_ids = []
        self.prev_text = ""
    
    def add_token(self, token_id: int) -> str:
        """Add a new token and return the text delta."""
        self.all_token_ids.append(token_id)
        full_text = self.tokenizer.decode(self.all_token_ids)
        # The delta is the new text that wasn't in the previous decode
        delta = full_text[len(self.prev_text):]
        self.prev_text = full_text
        return delta

class SSEFormatter:
    """Formats streaming responses as Server-Sent Events."""
    
    @staticmethod
    def format_chunk(request_id: str, delta_text: str, 
                     finish_reason: str = None) -> str:
        chunk = {
            "id": f"chatcmpl-{request_id}",
            "object": "chat.completion.chunk",
            "choices": [{
                "index": 0,
                "delta": {"content": delta_text} if delta_text else {},
                "finish_reason": finish_reason
            }]
        }
        return f"data: {json.dumps(chunk)}\n\n"
    
    @staticmethod
    def format_done() -> str:
        return "data: [DONE]\n\n"

# Simulate streaming generation
print("=== Simulated SSE Stream ===")
print("(Each line represents one SSE event sent to the client)\n")

detokenizer = IncrementalDetokenizer(tokenizer)
formatter = SSEFormatter()
request_id = "abc123"

# Simulate generating tokens
generated_tokens = [26, 27, 7, 6, 28, 29, 30, 1]  # simulated token IDs, ending with EOS

for i, token_id in enumerate(generated_tokens):
    delta = detokenizer.add_token(token_id)
    
    if token_id == tokenizer.eos_id:
        # Final chunk with finish_reason
        sse_event = formatter.format_chunk(request_id, "", finish_reason="stop")
    else:
        sse_event = formatter.format_chunk(request_id, delta)
    
    print(f"Token {i}: ID={token_id}, delta='{delta}'")
    print(f"  SSE: {sse_event.strip()}")

print(formatter.format_done().strip())
print(f"\nFull generated text: '{detokenizer.prev_text}'")

## 9. Complete Toy LLM Server

Let's now put all the pieces together into a complete (simulated) LLM serving system. This implementation shows the full request lifecycle with timing, state transitions, and all major components.

In [ ]:
import time
import random
from collections import deque

class ToyLLMServer:
    """A complete toy LLM server demonstrating the full request lifecycle."""
    
    def __init__(self, model_name="toy-llm", max_context=2048, 
                 max_batch_size=4, vocab_size=100):
        self.model_name = model_name
        self.max_context = max_context
        self.max_batch_size = max_batch_size
        self.vocab_size = vocab_size
        self.tokenizer = SimpleTokenizer()
        self.validator = RequestValidator(max_context_length=max_context,
                                          available_models=[model_name])
        
        # Server state
        self.waiting_queue = deque()
        self.running_batch = []
        self.completed_requests = []
        
        # Simulated timing (in seconds)
        self.validation_latency = 0.001
        self.tokenization_latency = 0.002
        self.prefill_latency_per_token = 0.0001  # 0.1ms per token
        self.decode_latency = 0.02  # 20ms per decode step
    
    def receive_request(self, request_body: dict) -> LLMRequest:
        """Stage 1: API Reception"""
        req = LLMRequest(
            request_id=str(uuid.uuid4())[:8],
            prompt=self._extract_prompt(request_body),
            max_tokens=request_body.get('max_tokens', 100),
            temperature=request_body.get('temperature', 1.0),
            stream=request_body.get('stream', False),
        )
        print(f"[{req.request_id}] Stage 1: Request received")
        return req
    
    def validate_request(self, req: LLMRequest, request_body: dict) -> bool:
        """Stage 2: Validation"""
        req.status = RequestStatus.VALIDATING
        time.sleep(self.validation_latency)
        
        is_valid, errors = self.validator.validate(request_body)
        req.validation_time = time.time()
        
        if not is_valid:
            req.status = RequestStatus.ERROR
            print(f"[{req.request_id}] Stage 2: Validation FAILED - {errors}")
            return False
        
        print(f"[{req.request_id}] Stage 2: Validation passed ({(req.validation_time - req.arrival_time)*1000:.1f}ms)")
        return True
    
    def tokenize_request(self, req: LLMRequest) -> None:
        """Stage 3: Tokenization"""
        req.status = RequestStatus.TOKENIZING
        time.sleep(self.tokenization_latency)
        
        req.token_ids = self.tokenizer.encode(req.prompt)
        req.tokenization_time = time.time()
        
        print(f"[{req.request_id}] Stage 3: Tokenized -> {len(req.token_ids)} tokens "
              f"({(req.tokenization_time - req.validation_time)*1000:.1f}ms)")
    
    def schedule_request(self, req: LLMRequest) -> None:
        """Stage 4: Add to scheduler queue"""
        req.status = RequestStatus.QUEUED
        req.scheduling_time = time.time()
        
        if len(self.running_batch) < self.max_batch_size:
            self.running_batch.append(req)
            print(f"[{req.request_id}] Stage 4: Scheduled immediately (batch size: {len(self.running_batch)})")
        else:
            self.waiting_queue.append(req)
            print(f"[{req.request_id}] Stage 4: Queued (position {len(self.waiting_queue)})")
    
    def prefill(self, req: LLMRequest) -> None:
        """Stage 5: Prefill - first forward pass"""
        req.status = RequestStatus.PREFILLING
        req.prefill_start_time = time.time()
        
        prefill_time = len(req.token_ids) * self.prefill_latency_per_token
        time.sleep(prefill_time)
        
        print(f"[{req.request_id}] Stage 5: Prefill complete ({len(req.token_ids)} tokens, "
              f"{prefill_time*1000:.1f}ms)")
    
    def decode_loop(self, req: LLMRequest) -> None:
        """Stage 6: Autoregressive decode loop"""
        req.status = RequestStatus.DECODING
        
        for step in range(req.max_tokens):
            time.sleep(self.decode_latency)
            
            # Simulate generating a token
            new_token = random.randint(5, self.vocab_size - 1)
            
            # Check for EOS (simulate 5% chance after 10 tokens)
            if step > 10 and random.random() < 0.05:
                new_token = self.tokenizer.eos_id
            
            req.generated_token_ids.append(new_token)
            
            if step == 0:
                req.first_token_time = time.time()
                print(f"[{req.request_id}] Stage 6: First token generated (TTFT={req.ttft*1000:.1f}ms)")
            
            if new_token == self.tokenizer.eos_id:
                print(f"[{req.request_id}] Stage 6: EOS at step {step+1}")
                break
        else:
            print(f"[{req.request_id}] Stage 6: max_tokens ({req.max_tokens}) reached")
    
    def detokenize_and_respond(self, req: LLMRequest) -> str:
        """Stage 7: Detokenization and response"""
        req.status = RequestStatus.DETOKENIZING
        req.generated_text = self.tokenizer.decode(req.generated_token_ids)
        req.status = RequestStatus.COMPLETE
        req.completion_time = time.time()
        
        print(f"[{req.request_id}] Stage 7: Response ready "
              f"(total latency={req.total_latency*1000:.1f}ms, "
              f"{len(req.generated_token_ids)} tokens generated)")
        return req.generated_text
    
    def process_request(self, request_body: dict) -> Optional[str]:
        """Process a request through the complete lifecycle."""
        print("\n" + "="*60)
        
        # Stage 1: Reception
        req = self.receive_request(request_body)
        
        # Stage 2: Validation
        if not self.validate_request(req, request_body):
            return None
        
        # Stage 3: Tokenization
        self.tokenize_request(req)
        
        # Stage 4: Scheduling
        self.schedule_request(req)
        
        # Stage 5: Prefill
        self.prefill(req)
        
        # Stage 6: Decode
        self.decode_loop(req)
        
        # Stage 7: Detokenize & Respond
        result = self.detokenize_and_respond(req)
        
        self.completed_requests.append(req)
        return result
    
    def _extract_prompt(self, request_body: dict) -> str:
        if 'messages' in request_body:
            return self.tokenizer.apply_chat_template(request_body['messages'])
        return request_body.get('prompt', '')

# Run the server!
server = ToyLLMServer(model_name="toy-llm", max_batch_size=4)

request = {
    "model": "toy-llm",
    "messages": [
        {"role": "user", "content": "explain quantum computing in simple terms"}
    ],
    "max_tokens": 30,
    "temperature": 0.7,
    "stream": True
}

result = server.process_request(request)

## 10. Visualizing the Request Timeline

Let's visualize the timing of each stage for multiple requests to understand how the pipeline works.

In [ ]:
# Process multiple requests and visualize timelines
server2 = ToyLLMServer(model_name="toy-llm", max_batch_size=4)

requests_data = [
    {"model": "toy-llm", "messages": [{"role": "user", "content": "short prompt"}], "max_tokens": 15},
    {"model": "toy-llm", "messages": [{"role": "user", "content": "this is a medium length prompt that has more tokens"}], "max_tokens": 20},
    {"model": "toy-llm", "messages": [{"role": "user", "content": "a very long prompt " * 5}], "max_tokens": 25},
]

for r in requests_data:
    server2.process_request(r)

# Visualize
fig, ax = plt.subplots(figsize=(16, 6))

stage_colors = {
    'Validation': '#3498db',
    'Tokenization': '#2ecc71', 
    'Scheduling': '#9b59b6',
    'Prefill': '#e74c3c',
    'Decode': '#f39c12',
    'Detokenize': '#1abc9c',
}

for i, req in enumerate(server2.completed_requests):
    base_time = server2.completed_requests[0].arrival_time
    y = len(server2.completed_requests) - 1 - i
    
    stages = [
        ('Validation', req.arrival_time - base_time, req.validation_time - base_time),
        ('Tokenization', req.validation_time - base_time, req.tokenization_time - base_time),
        ('Scheduling', req.tokenization_time - base_time, req.prefill_start_time - base_time),
        ('Prefill', req.prefill_start_time - base_time, req.first_token_time - base_time),
        ('Decode', req.first_token_time - base_time, req.completion_time - base_time - 0.001),
        ('Detokenize', req.completion_time - base_time - 0.001, req.completion_time - base_time),
    ]
    
    for stage_name, start, end in stages:
        duration = end - start
        ax.barh(y, duration, left=start, height=0.5, 
                color=stage_colors[stage_name], alpha=0.8, edgecolor='black', linewidth=0.5)
    
    # Add TTFT marker
    ttft_time = req.first_token_time - base_time
    ax.plot(ttft_time, y, 'v', color='red', markersize=10, zorder=5)
    ax.text(req.completion_time - base_time + 0.01, y, 
            f'{len(req.generated_token_ids)} tokens, TTFT={req.ttft*1000:.0f}ms', 
            va='center', fontsize=9)

ax.set_yticks(range(len(server2.completed_requests)))
ax.set_yticklabels([f'Request {req.request_id}' for req in reversed(server2.completed_requests)])
ax.set_xlabel('Time (seconds from first request)')
ax.set_title('Request Lifecycle Timeline', fontsize=14, fontweight='bold')

# Legend
patches = [mpatches.Patch(color=c, label=n) for n, c in stage_colors.items()]
patches.append(plt.Line2D([0], [0], marker='v', color='red', linestyle='', markersize=10, label='TTFT'))
ax.legend(handles=patches, loc='upper right', ncol=3)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Connection Management

In production, LLM servers must handle real-world connection issues:

### Keep-Alive
HTTP keep-alive allows reusing TCP connections for multiple requests, reducing connection establishment overhead. For streaming responses, the connection must remain open for the duration of generation.

### Timeouts
Multiple timeout levels are needed:
- **Connection timeout**: How long to wait for initial TCP handshake
- **Request timeout**: Max time for the entire request (guards against infinite generation)
- **Idle timeout**: Close connection if no data flows for too long

### Request Cancellation
When a client disconnects mid-stream:
1. Detect the broken connection (TCP RST or write error)
2. Signal the engine to stop generating for this request
3. Free the KV-cache blocks allocated to this request
4. Remove from the running batch to make room for other requests

This is critical for efficiency: a cancelled request consuming GPU cycles and memory is pure waste.

In [ ]:
import asyncio

class ConnectionManager:
    """Manages client connections with timeout and cancellation support."""
    
    def __init__(self, request_timeout: float = 30.0, idle_timeout: float = 5.0):
        self.request_timeout = request_timeout
        self.idle_timeout = idle_timeout
        self.active_connections: Dict[str, dict] = {}
    
    def register(self, request_id: str):
        """Register a new connection."""
        self.active_connections[request_id] = {
            'start_time': time.time(),
            'last_activity': time.time(),
            'cancelled': False,
            'tokens_sent': 0,
        }
        print(f"  [ConnMgr] Connection registered: {request_id}")
    
    def heartbeat(self, request_id: str):
        """Update last activity time (called when data is sent)."""
        if request_id in self.active_connections:
            self.active_connections[request_id]['last_activity'] = time.time()
            self.active_connections[request_id]['tokens_sent'] += 1
    
    def cancel(self, request_id: str):
        """Cancel a request (e.g., client disconnected)."""
        if request_id in self.active_connections:
            self.active_connections[request_id]['cancelled'] = True
            print(f"  [ConnMgr] Connection cancelled: {request_id}")
    
    def is_cancelled(self, request_id: str) -> bool:
        conn = self.active_connections.get(request_id)
        if not conn:
            return True
        if conn['cancelled']:
            return True
        
        now = time.time()
        # Check request timeout
        if now - conn['start_time'] > self.request_timeout:
            print(f"  [ConnMgr] Request timeout: {request_id}")
            return True
        return False
    
    def close(self, request_id: str):
        """Clean up connection resources."""
        if request_id in self.active_connections:
            conn = self.active_connections.pop(request_id)
            duration = time.time() - conn['start_time']
            print(f"  [ConnMgr] Connection closed: {request_id} "
                  f"(duration={duration:.2f}s, tokens={conn['tokens_sent']})")

# Demonstrate connection management
conn_mgr = ConnectionManager(request_timeout=1.0)  # Short timeout for demo

# Normal request
conn_mgr.register("req-001")
for i in range(5):
    time.sleep(0.05)
    conn_mgr.heartbeat("req-001")
conn_mgr.close("req-001")

print()

# Cancelled request (simulating client disconnect)
conn_mgr.register("req-002")
conn_mgr.heartbeat("req-002")
conn_mgr.cancel("req-002")  # Client disconnects!
print(f"  Is req-002 cancelled? {conn_mgr.is_cancelled('req-002')}")
conn_mgr.close("req-002")

print()

# Timeout request
conn_mgr.register("req-003")
time.sleep(1.1)  # Exceed timeout
print(f"  Is req-003 timed out? {conn_mgr.is_cancelled('req-003')}")
conn_mgr.close("req-003")

## 12. Key Performance Metrics

Understanding performance requires tracking several key metrics:

| Metric | Definition | Affected By |
|--------|-----------|-------------|
| **TTFT** (Time to First Token) | Time from request arrival to first generated token | Prompt length, queue wait, prefill speed |
| **TPOT** (Time Per Output Token) | Average time between consecutive output tokens | Batch size, KV-cache size, model size |
| **Throughput** (tokens/sec) | Total tokens generated per second across all requests | Batching efficiency, GPU utilization |
| **TTLT** (Time to Last Token) | Total end-to-end latency | TTFT + (num_tokens * TPOT) |
| **Goodput** | Throughput that meets SLO requirements | Scheduling policy, load balancing |

In [ ]:
# Visualize the relationship between metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: TTFT breakdown
ax = axes[0]
categories = ['Queue\nWait', 'Validation +\nTokenization', 'Prefill\n(short prompt)', 'Prefill\n(long prompt)']
short_ttft = [5, 2, 10, 0]  # ms
long_ttft = [50, 2, 0, 200]  # ms

x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, short_ttft, width, label='Short prompt (128 tokens)', color='#3498db', alpha=0.8)
ax.bar(x + width/2, long_ttft, width, label='Long prompt (4096 tokens)', color='#e74c3c', alpha=0.8)
ax.set_ylabel('Time (ms)')
ax.set_title('TTFT Breakdown', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Throughput vs Latency tradeoff
ax = axes[1]
batch_sizes = [1, 2, 4, 8, 16, 32, 64]
# Simulated: throughput increases with batch size, but so does per-request latency
throughput = [50, 95, 180, 330, 580, 900, 1200]  # tokens/sec
avg_tpot = [20, 21, 22, 24, 28, 36, 53]  # ms

ax.plot(batch_sizes, throughput, 'o-', color='#2ecc71', linewidth=2, markersize=8, label='Throughput (tokens/s)')
ax2 = ax.twinx()
ax2.plot(batch_sizes, avg_tpot, 's-', color='#e74c3c', linewidth=2, markersize=8, label='TPOT (ms)')

ax.set_xlabel('Batch Size')
ax.set_ylabel('Throughput (tokens/sec)', color='#2ecc71')
ax2.set_ylabel('Time Per Output Token (ms)', color='#e74c3c')
ax.set_title('Throughput vs Latency Tradeoff', fontweight='bold')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 13. How vLLM and SGLang Implement the Lifecycle

Now that we understand the conceptual lifecycle, let's briefly map it to real implementations:

### vLLM Architecture

```
HTTP Request
  -> api_server.py (FastAPI + uvicorn)
    -> AsyncLLMEngine.add_request()
      -> Tokenizer.encode()
      -> Scheduler.add_seq_group()
        -> Scheduler.schedule() [called each iteration]
          -> ModelRunner.execute_model()
            -> Sampler.sample()
              -> Detokenizer.decode()
                -> SSE stream or JSON response
```

### SGLang Architecture

```
HTTP Request
  -> launch_server.py (FastAPI)
    -> TokenizerManager.handle_request()
      -> Tokenizer.encode()
      -> Scheduler.add_request() [via ZMQ]
        -> Scheduler.get_next_batch_to_run()
          -> ModelRunner.forward_batch()
            -> Sampler.sample()
              -> Detokenizer.decode() [in separate process]
                -> SSE stream or JSON response
```

### Key Difference
SGLang uses **RadixAttention** for automatic prefix caching, which can skip prefill for shared prompt prefixes. This is a major optimization we'll cover in Chapter 2.3.

## 14. Exercises

### Exercise 1: Trace a Request
Given the following request, trace it through all 7 stages and calculate estimated timing:
- Model: Llama-3-8B (32 layers, 4096 hidden dim)
- Prompt: 256 tokens
- max_tokens: 100
- Running on A100-80GB with ~312 TFLOPS (FP16)

Estimate: TTFT, TPOT, and total latency.

In [ ]:
# Exercise 1: Fill in the calculations

# Model parameters
num_layers = 32
hidden_dim = 4096
num_heads = 32
head_dim = hidden_dim // num_heads  # 128
vocab_size = 128256  # Llama-3 vocab
prompt_tokens = 256
max_gen_tokens = 100

# GPU specs
gpu_tflops = 312  # A100 FP16 TFLOPS
gpu_memory_bandwidth = 2039  # GB/s for A100-80GB

# YOUR CODE: Calculate prefill FLOPs per layer
# Hint: Consider QKV projection, attention, output projection, and FFN
prefill_flops_per_layer = (
    prompt_tokens * hidden_dim * 3 * hidden_dim +  # QKV
    2 * prompt_tokens * prompt_tokens * hidden_dim +  # Attention
    prompt_tokens * hidden_dim * hidden_dim +  # Output projection
    2 * prompt_tokens * hidden_dim * 4 * hidden_dim  # FFN (up + down)
)

total_prefill_flops = prefill_flops_per_layer * num_layers
estimated_prefill_ms = total_prefill_flops / (gpu_tflops * 1e12) * 1000

print(f"=== Exercise 1: Request Timing Estimation ===")
print(f"Prefill FLOPs per layer: {prefill_flops_per_layer/1e9:.2f} GFLOPs")
print(f"Total prefill FLOPs: {total_prefill_flops/1e12:.4f} TFLOPs")
print(f"Estimated prefill time: {estimated_prefill_ms:.2f} ms")
print(f"\nEstimated TTFT (prefill + overhead): ~{estimated_prefill_ms + 5:.0f} ms")

# YOUR CODE: Calculate decode step FLOPs and time
# Hint: For decode, the bottleneck is memory bandwidth (loading model weights)
model_size_bytes = 8e9 * 2  # 8B parameters in FP16
decode_time_ms = model_size_bytes / (gpu_memory_bandwidth * 1e9) * 1000
print(f"\nEstimated TPOT (memory-bound): ~{decode_time_ms:.1f} ms")
print(f"Estimated total decode time: ~{decode_time_ms * max_gen_tokens:.0f} ms")
print(f"\nEstimated total latency: ~{estimated_prefill_ms + 5 + decode_time_ms * max_gen_tokens:.0f} ms")

### Exercise 2: Implement Cancellation

Modify the `ToyLLMServer` to support request cancellation during the decode loop. When a request is cancelled:
1. Stop generating immediately
2. Return partial results
3. Log the cancellation with timing

In [ ]:
# Exercise 2: Implement cancellation support
# Hint: Add a cancel_after parameter to simulate client disconnect

class ToyLLMServerWithCancellation(ToyLLMServer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conn_mgr = ConnectionManager()
    
    def decode_loop_with_cancel(self, req: LLMRequest, cancel_after: int = None) -> None:
        """YOUR CODE: Implement decode loop with cancellation support.
        
        If cancel_after is set, simulate client disconnecting after that many tokens.
        """
        req.status = RequestStatus.DECODING
        self.conn_mgr.register(req.request_id)
        
        for step in range(req.max_tokens):
            # TODO: Check if request is cancelled
            # TODO: Simulate cancellation after 'cancel_after' tokens
            # TODO: Handle EOS token
            # TODO: Generate and append token
            pass  # Replace with your implementation
        
        self.conn_mgr.close(req.request_id)

print("Exercise 2: Implement the decode_loop_with_cancel method above.")
print("Test it with cancel_after=5 to simulate a client disconnecting after 5 tokens.")

### Exercise 3: Batch Processing Efficiency

Compare the GPU utilization of static batching vs continuous batching by simulating 10 requests with varying generation lengths.

In [ ]:
# Exercise 3: Simulate and compare batching strategies

def simulate_static_batching(requests: List[dict], batch_size: int = 4):
    """Simulate static batching. Returns (total_time, gpu_utilization).
    
    YOUR CODE: Implement static batching simulation.
    In static batching, a batch waits for ALL requests to finish before starting the next.
    """
    pass

def simulate_continuous_batching(requests: List[dict], batch_size: int = 4):
    """Simulate continuous batching. Returns (total_time, gpu_utilization).
    
    YOUR CODE: Implement continuous batching simulation.
    In continuous batching, finished requests are replaced immediately.
    """
    pass

# Test data: 10 requests with varying generation lengths
test_requests = [
    {'prompt_tokens': 100, 'gen_tokens': random.randint(10, 200)}
    for _ in range(10)
]

print("Exercise 3: Implement both batching simulations and compare their efficiency.")
print(f"\nTest requests (prompt_tokens, gen_tokens):")
for i, r in enumerate(test_requests):
    print(f"  Request {i}: {r['prompt_tokens']} prompt tokens, {r['gen_tokens']} gen tokens")

## 15. Summary

In this chapter, we traced the complete lifecycle of a request through an LLM serving system:

1. **API Reception**: HTTP request arrives, parsed by async web framework
2. **Validation**: Authentication, parameter bounds, context length, rate limiting
3. **Tokenization**: Text to token IDs via BPE/SentencePiece, chat template application
4. **Scheduling**: Determines when and how requests get GPU time (continuous batching)
5. **Prefill**: Process all input tokens at once (compute-bound, generates KV-cache)
6. **Decode**: Generate one token per iteration (memory-bound, autoregressive)
7. **Detokenization & Streaming**: Convert tokens back to text, stream via SSE

### Key Takeaways

- **Prefill is compute-bound**, decode is **memory-bound** -- they have fundamentally different optimization strategies
- **Continuous batching** dramatically improves throughput over static batching
- **Connection management** (timeouts, cancellation) is critical for resource efficiency
- **TTFT and TPOT** are the two most important latency metrics

## What's Next?

In **Chapter 2.2: Scheduler Design**, we'll dive deep into the scheduling algorithms that make continuous batching possible, including FCFS, priority-based scheduling, preemption strategies, and how to meet SLO constraints.